### Installation and Import of Modules

In [0]:
%pip install pybaseball -q
%pip install sqlalchemy psycopg2-binary -q

### Postgres Imports
import psycopg2
import sqlalchemy
from psycopg2.extras import execute_batch
from sqlalchemy import create_engine, text

### PyBaseball (Module to get Statcast Data) Imports
from pybaseball import (
    statcast, statcast_batter, statcast_pitcher, statcast_pitcher_spin,
    statcast_fielding, statcast_running, playerid_lookup, playerid_reverse_lookup,
    statcast_pitcher_exitvelo_barrels, statcast_pitcher_expected_stats,
    statcast_pitcher_pitch_arsenal, statcast_pitcher_arsenal_stats,
    statcast_pitcher_percentile_ranks, statcast_pitcher_spin_dir_comp,
    statcast_outs_above_average, statcast_catcher_framing,
    statcast_outfield_directional_oaa, statcast_outfield_catch_prob,
    statcast_outfielder_jump, statcast_catcher_poptime,
    statcast_sprint_speed, statcast_running_splits,
    playerid_reverse_lookup
)

### Spark Imports
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import col, avg, count, desc, pandas_udf
from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType, IntegerType, DoubleType, BooleanType, LongType
)

### Other Imports
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from io import StringIO
from itertools import combinations
from typing import Optional, Union

import csv
import json
import logging
import os
import pandas as pd
import requests
import shutil
import time
import warnings

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


#### Defining Important Variables

In [0]:
batter_directory = "***"
pitcher_directory = "***"
train_data_directory = "***"

### Defining a Logger

In [0]:
class DBFSFlushHandler(logging.Handler):
    def __init__(self, local_path, dbfs_path):
        super().__init__()
        self.local_path = local_path
        self.dbfs_path = dbfs_path
        self.log_file = open(local_path, "a")

    def emit(self, record):
        try:
            msg = self.format(record)
            self.log_file.write(msg + "\n")
            self.log_file.flush()
            shutil.copy(self.local_path, self.dbfs_path)
        except Exception:
            self.handleError(record)

    def close(self):
        self.log_file.close()
        super().close()

#### Clearing any Pre-Existing Loggers

In [0]:
try:
    logger = logging.getLogger("statcast_logger")
    for handler in logger.handlers:
        handler.close()
        logger.removeHandler(handler)
except Exception as e:
    print("Logger shutdown failed:", e)

for var in ["statcast_logger", "statcast_log_file"]:
    if var in globals():
        del globals()[var]

import os

paths = [
    "***.txt",
    "***.txt"
]

for path in paths:
    if os.path.exists(path):
        os.remove(path)

#### Initializing the Logger

In [0]:
# Run this once to initialize
if 'statcast_logger' not in globals():
    local_path = "***.txt"
    dbfs_path = "***.txt"

    logger = logging.getLogger("statcast_logger")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    if logger.hasHandlers():
        logger.handlers.clear()

    handler = DBFSFlushHandler(local_path, dbfs_path)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

    statcast_logger = logger

In [0]:
def write_log(log):
    statcast_logger.info(log)

def log_and_print(log):
    statcast_logger.info(log)
    print(log)

### Postgres Configuration and Connection

In [0]:
jdbc_url = "***"
rds_endpoint = "***"

connection_properties = {
    "user": "***",
    "password": "$***",
    "driver": "***",
    "port": "***"
}

username = connection_properties['user']
password = connection_properties['password']
port = connection_properties['port']
database_name = 'statcast'

engine = create_engine(
    f'postgresql://{username}:{password}@{rds_endpoint}:{port}/{database_name}'
)

In [0]:
def get_conn():
    return psycopg2.connect(
        host=rds_endpoint,
        database=database_name,
        user=username,
        password=password,
        port=port
    )

#### Postgres Utility Functions

In [0]:
def create_rds_table(table_name: str, df: pd.DataFrame):
    df.head(0).to_sql(con=engine, name=table_name, if_exists='replace', index=False)

def get_database_table_sizes():
    tables_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("query", "SELECT relname AS table_name, pg_size_pretty(pg_total_relation_size(relid)) AS total_size FROM pg_catalog.pg_statio_user_tables") \
    .option("user", connection_properties.get("user")) \
    .option("password", connection_properties.get("password")) \
    .load()

    tables_df.show(truncate=False)

def read_from_rds(query: str) -> DataFrame:
    return spark.read.jdbc(
        url = jdbc_url,
        table = query,
        properties=connection_properties
    )

### Create a Spark Session

In [0]:
spark = SparkSession.builder.appName("StatcastFetcher").getOrCreate()

### Statcast Utility Functions

In [0]:
def get_statcast(year: int):
    df = statcast(start_dt=f'{year}-03-01', end_dt=f'{year}-12-01')
    df['game_date'] = df['game_date'].astype(str)
    df['year'] = year
    return df

In [0]:
def get_statcast_batter_pitcher_year(year, player_id, statcast_type, retries=3, backoff=2):
    retry = 0
    while retry <= retries:
        try:
            start_dt = f'{year}-03-01'
            end_dt = f'{year}-12-01'
            if statcast_type == 'batter':
                df = statcast_batter(start_dt, end_dt, player_id)
            else:
                df = statcast_pitcher(start_dt, end_dt, player_id)

            if retry > 0:
                checks = "✅ " * (retry + 1)
                log_and_print(f"{checks}[{datetime.now()}] On retry attempt {retry}, successfully fetched data for {statcast_type} {player_id} for {year}")

            return df
        except Exception as e:
            retry += 1
            log_and_print(f"[{datetime.now()}] ❌ Attempt {retry}: Failed to fetch data for {statcast_type} {player_id} for {year}: {e}")

            if retry >= retries:
                log_and_print(f"[{datetime.now()}] ❌ Giving up on {statcast_type} {player_id} for {year} after {retry} attempts.")
                raise e

            # Exponential backoff delay
            sleep_time = backoff * (2 ** (retry - 1))
            log_and_print(f"[{datetime.now()}] ⏳ Retrying in {sleep_time} seconds...")
            time.sleep(sleep_time)

In [0]:
### All Code in this cell is copied from https://github.com/jldbc/pybaseball/blob/master/pybaseball/utils.py

pitch_codes = ["FF", "CU", "CH", "FC", "EP", "FO", "KN", "KC", "SC", "SI", "SL", "FS", "FT", "ST", "SV", "SIFT", "CUKC"]
pitch_names = ["4-Seamer", "Curveball", "Changeup", "Cutter", "Eephus", "Forkball", "Knuckleball", "Knuckle-curve", "Screwball", "Sinker", "Slider", "Splitter", "2-Seamer", "Sweeper", "Slurve", "Sinker", "Curveball"]
pitch_names_upper = [p.upper() for p in pitch_names]

pitch_name_to_code_map = dict(zip(pitch_codes + pitch_names_upper, pitch_codes + pitch_codes))
pitch_code_to_name_map = dict(zip(pitch_codes, pitch_names))

position_codes = ["IF", "OF", "C", "1B", "2B", "3B", "SS", "LF", "CF", "RF", "ALL"]
position_names = ["Infield", "Outfield", "Catcher", "First Base", "Second Base", "Third Base", "Shortstop", "Left Field", "Center Field", "Right Field"]
position_names_upper = [p.upper() for p in position_names]

pos_code_to_numbers_map = dict(zip(position_codes[2:10], [str(x) for x in range(2, 10)]))
pos_name_to_code_map = dict(zip(position_codes + position_names_upper, position_codes + position_codes))
pos_code_to_name_map = dict(zip(position_codes, position_names))

def norm_pitch_code(pitch: str, to_word: bool = False) -> str:
	normed = pitch_name_to_code_map.get(pitch.upper())
	normed = pitch_code_to_name_map.get(normed) if to_word and normed else normed
	if normed is None:
		if pitch.lower() == 'all':
			raise ValueError("'All' is not a valid pitch in this particular context!")
		raise ValueError(f'{pitch} is not a valid pitch!')
	return normed

def sanitize_statcast_columns(df: pd.DataFrame) -> pd.DataFrame:
	'''
	Creates uniform structure in Statcast column names
	Removes leading whitespace in column names
	'''
	df.columns = df.columns.str.strip()
	return df

def norm_positions(pos: Union[int, str], to_word: bool = False, to_number: bool = True) -> str:
	pos_str = str(pos)
	normed: Optional[str] = None
	if pos_str in pos_code_to_numbers_map.values():
		to_number = False
		normed = pos_str
	else:
		normed = pos_name_to_code_map.get(pos_str.upper())
		normed = pos_code_to_name_map.get(normed) if to_word and normed else normed
	if to_number:
		if normed not in ["IF", "OF"]:
			normed = pos_code_to_numbers_map.get(normed) if normed else normed
		if pos_str.lower() == "all":
			normed = ""
	if normed is None:
		raise ValueError(f'{pos} is not a valid position!')
	# lower() ok due to positional numbers being cast as strings when created
	return normed.lower()

### RDS Uitility Functions

#### Functions to Prepare Pandas DataFrame for RDS Upload

In [0]:
def quote_strings_with_commas(df: pd.DataFrame) -> pd.DataFrame:
    df_quoted = df.copy()
    for col in df_quoted.select_dtypes(include=['string', 'object']):
        if df_quoted[col].apply(lambda x: isinstance(x, str) and ',' in x).any():
            df_quoted[col] = df_quoted[col].apply(
                lambda x: f'"{x}"' if isinstance(x, str) and ',' in x else x
            )
    return df_quoted

In [0]:
def convert_lists_to_json(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()
    for col in df_clean.columns:
        if df_clean[col].apply(lambda x: isinstance(x, (list, dict, set))).any():
            df_clean[col] = df_clean[col].apply(
                lambda x: json.dumps(x) if isinstance(x, (list, dict, set)) else x
            )
    return df_clean

In [0]:
def clean_dataframe_for_postgres(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()

    for col in df_clean.columns:
        dtype = df_clean[col].dtype

        if pd.api.types.is_float_dtype(dtype):
            if (df_clean[col].dropna() % 1 == 0).all():
                df_clean[col] = df_clean[col].astype('Int64')
            else:
                df_clean[col] = df_clean[col].astype('float64')

        elif pd.api.types.is_integer_dtype(dtype):
            df_clean[col] = df_clean[col].astype('Int64')

        elif pd.api.types.is_bool_dtype(dtype):
            df_clean[col] = df_clean[col].astype('boolean')

        elif pd.api.types.is_object_dtype(dtype) or pd.api.types.is_string_dtype(dtype):
            non_null = df_clean[col].dropna()
            all_numeric = pd.to_numeric(non_null, errors="coerce").notna().all()

            if all_numeric:
                df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
            else:
                df_clean[col] = (
                    df_clean[col]
                    .astype(str)
                    .replace({r'[\n\r]+': ' ', r'"': "'", r',': ';', r'\\': '/'}, regex=True)
                )

        elif pd.api.types.is_datetime64_any_dtype(dtype):
            df_clean[col] = pd.to_datetime(df_clean[col])

    return df_clean


#### Copy Data to the Postgres Database

In [0]:
def get_numeric_columns(df: pd.DataFrame):
    return df.select_dtypes(include='number').columns.tolist()

In [0]:
def fast_copy_to_sql(df: pd.DataFrame, table_name: str, conn):
    try:
        df = clean_dataframe_for_postgres(df)
        df = convert_lists_to_json(df)
        df.replace([pd.NA, float('inf'), float('-inf'), ''], None, inplace=True)

        for col in get_numeric_columns(df):
            if col in df.columns:
                df[col] = df[col].apply(
                    lambda x: None if pd.isna(x) or str(x).strip().lower() in ['', 'nan', 'n/a'] else x
                )
                df[col] = pd.to_numeric(df[col], errors='coerce')

                if (df[col].dropna() % 1 == 0).all():
                    df[col] = df[col].astype('Int64')
                else:
                    df[col] = df[col].astype('float64')


        df = quote_strings_with_commas(df)

        buffer = StringIO()
        for col in df.columns:
            if df[col].apply(lambda x: isinstance(x, (list, dict, set))).any():
                log_and_print(f"⚠️ Column {col} contains unflattened nested values")
        df.to_csv(
            buffer,
            index=False,
            header=False,
            na_rep='',
            quoting=csv.QUOTE_MINIMAL,
            escapechar='\\'
        )
        buffer.seek(0)

        columns = ', '.join(f'"{col}"' for col in df.columns)
        copy_command = f"COPY {table_name} ({columns}) FROM STDIN WITH CSV NULL ''"

        cursor = conn.cursor()
        cursor.copy_expert(
            sql=copy_command,
            file=buffer
        )
        conn.commit()
        log_and_print(f"[{datetime.now()}] ✅ Uploaded {len(df)} rows to {table_name}")

    except Exception as e:
        conn.rollback()
        log_and_print(f"[{datetime.now()}] ❌ COPY failed: {e}")
        log_and_print(df.head(5).to_string())

    finally:
        if 'cursor' in locals():
            cursor.close()

#### Functions to Upload Data to RDS

In [0]:
#statcast type can be batter, pitcher
def upload_batter_pitcher_batch(batch, statcast_type):
    statcast_type = statcast_type.lower() 
    try:
        dfs = []
        for row in batch:
            year = row['year']
            player_id = row[statcast_type]

            log_and_print(f"[{datetime.now()}] ⬇️ Fetching data for {statcast_type} {player_id} - {year}")
            df = get_statcast_batter_pitcher_year(year, player_id, statcast_type)
            log_and_print(f"[{datetime.now()}] ⬆️ Fetched data for {statcast_type} {player_id} - {year}")

            df['year'] = year
            dfs.append(df)

        combined_df = pd.concat(dfs)
        log_and_print(f"[{datetime.now()}] ⬆️ Uploading batch of {len(batch)} {statcast_type}s")

        conn = psycopg2.connect(
            host=rds_endpoint,
            database=database_name,
            user=username,
            password=password,
            port=port
        )

        fast_copy_to_sql(combined_df, f'statcast_{statcast_type}s', conn)
        conn.close()
        return f"✅ Uploaded {len(batch)} pitchers"

    except Exception as e:
        error_string = f"❌ Batch failed | Error: {str(e)}"
        log_and_print(error_string)
        return error_string

In [0]:
def group_rows(rows, batch_size=5):
    for i in range(0, len(rows), batch_size):
        yield rows[i:i+batch_size]

def upload_data_to_rds_in_parallel(spark_df, statcast_type, max_workers=2, batch_size=5):
    rows = spark_df.collect()
    batches = list(group_rows(rows, batch_size=batch_size))

    log_and_print("Uploading {statcast_type} data to RDS.")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(upload_batter_pitcher_batch, batch, statcast_type) for batch in batches]
        for future in as_completed(futures):
            log_and_print(future.result())

#### Other Utility Functions

In [0]:
def list_tables():
    tables_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("query", "SELECT schemaname, tablename FROM pg_catalog.pg_tables WHERE schemaname NOT IN ('pg_catalog', 'information_schema')") \
    .option("user", connection_properties.get("user")) \
    .option("password", connection_properties.get("password")) \
    .load()

    return tables_df

In [0]:
def get_table_schema(tablename: str):
    query = f"(select column_name, data_type, is_nullable, column_default from information_schema.columns where table_name = '{tablename}' order by ordinal_position) as schema_query"
    return read_from_rds(query)

### Retrieving Statcast Batter and Pitcher IDs

We need to retrieve the ids of batters and pitchers to call other statcast functions and download statcast data in more granular tables. This can be done by calling the main statcast function, statcast(), and from those results, retrieve the batter and pitcher ids.

In [0]:
def upload_statcast_main_to_rds(years=[x for x in range(2015, 2025)]):
    for year in years:
        df = get_statcast(year)
        spark_df = spark.createDataFrame(df)
        mode = "append"
        if year == 2015:
            mode = "overwrite"
            create_rds_table(statcast, df)

        spark_df.write.jdbc(url=jdbc_url, table="statcast", mode=mode, properties=connection_properties)

def get_batter_pitcher_ids_from_rds():
    batters = read_from_rds("SELECT DISTINCT year, batter FROM statcast")
    pitchers = read_from_rds("SELECT DISTINCT year, pitcher FROM statcast")

    return batters, pitchers

def write_batters_pitchers_to_parquet():
    batters, pitchers = get_batter_pitcher_ids_from_rds()

    batters.write.parquet(batter_directory)
    pitchers.write.parquet(pitcher_directory)

In [0]:
#The following only needs to be uncommented and run at the beginning when the batter and pitcher ids are unknown
#upload_statcast_main_to_rds()

#Checkpointing the data, only need to uncomment if the above function call is uncommented
#write_batters_pitchers_to_parquet()

### Retrieve Batter and Pitcher IDs from Parquet

In [0]:
def get_batter_pitcher_ids_from_parquet():
    batters = spark.read.parquet(batter_directory)
    pitchers = spark.read.parquet(pitcher_directory)

    return batters, pitchers

In [0]:
batters, pitchers = get_batter_pitcher_ids_from_parquet()
batters = batters.orderBy(['year', 'batter'])
pitchers = pitchers.orderBy(['year', 'pitcher'])

### Creating RDS Tables

In [0]:
def get_pitcher_pitcher_arsenal_table():
    df1 = statcast_pitcher_pitch_arsenal(2015, minP=1, arsenal_type="n_")
    df2 = statcast_pitcher_pitch_arsenal(2015, minP=1, arsenal_type="avg_speed")
    df3 = statcast_pitcher_pitch_arsenal(2015, minP=1, arsenal_type="avg_spin")
    return df1.merge(df2, on=["last_name, first_name", "pitcher"]).merge(df3, on=["last_name, first_name", "pitcher"])

In [0]:
def get_fielding_run_value_schema_df():
    df = statcast_fielding_run_value(2020, pos=3).head()
    df['run_value_catching'] = df['run_value_catching'].astype('float64')
    return df

In [0]:
def create_rds_tables():
  create_rds_table('statcast_batters', get_statcast_batter_pitcher_year(2015, 115629, 'batter'))
  create_rds_table('statcast_pitchers', get_statcast_batter_pitcher_year(2015, 518774, 'pitcher'))
  create_rds_table('batter_exitvelo_barrels', statcast_batter_exitvelo_barrels(2020, minBBE=1).head())
  create_rds_table('batter_expected_stats', statcast_batter_expected_stats(2020,minPA=1).head())
  create_rds_table('batter_percentile_ranks', statcast_batter_percentile_ranks(2020).head())
  create_rds_table('batter_pitcher_arsenal', statcast_batter_pitch_arsenal(2020, minPA=1).head())
  create_rds_table('pitcher_exitvelo_barrels',statcast_pitcher_exitvelo_barrels(2020, minBBE=1).head())
  create_rds_table('pitcher_expected_stats',statcast_pitcher_expected_stats(2019, minPA=1).head())
  create_rds_table('pitcher_pitch_arsenal',get_pitcher_pitcher_arsenal_table().head())
  create_rds_table('pitcher_arsenal_stats', statcast_pitcher_arsenal_stats(2018, minPA=1).head())
  create_rds_table('pitcher_percentile_ranks', statcast_pitcher_percentile_ranks(2015).head())
  create_rds_table('pitch_movement', statcast_pitcher_pitch_movement(2015).head())
  create_rds_table('active_spin', statcast_pitcher_active_spin(2015).head())
  create_rds_table('oaa', statcast_outs_above_average(2015, 3).head())
  create_rds_table('directional_oaa', statcast_outfield_directional_oaa(2020).head())
  create_rds_table('outfield_catch_prob', statcast_outfield_catch_prob(2020).head())
  create_rds_table('outfield_jump', statcast_outfielder_jump(2020).head())
  create_rds_table('catcher_poptime', statcast_catcher_poptime(2020).head())
  create_rds_table('catcher_framing', statcast_catcher_framing(2020).head())
  create_rds_table('fielding_run_value', get_fielding_run_value_schema_df().head())
  create_rds_table('sprint_speed', statcast_sprint_speed(2020).head())
  create_rds_table('running_splits', statcast_running_splits(2020).head())
#create_rds_tables()

### Uploading Batter Data to RDS

In [0]:
def upload_batters_to_rds(years=[x for x in range(2015,2025)]):
    for year in years:
        upload_data_to_rds_in_parallel(batters.where(f"year == {year}"), 'batter')

#upload_batters_to_rds()

### Uploading Pitcher Data to RDS

In [0]:
def upload_pitchers_to_rds(years=[x for x in range(2015, 2025)]):
    for year in years:
        upload_data_to_rds_in_parallel(pitchers.where(f"year == {year}"), 'pitcher')

#upload_pitchers_to_rds()

### Other Statcast Batter Data

In [0]:
from pybaseball import (
    statcast_batter_exitvelo_barrels, statcast_batter_expected_stats,
    statcast_batter_percentile_ranks, statcast_batter_pitch_arsenal
)

#### Uploading Batter Exit Velocity and Barrels Data to RDS

In [0]:
def upload_batter_exitvelo_barrels_to_rds(years=[x for x in range(2015,2025)], minBBE=1):
    table = 'batter_exitvelo_barrels'

    conn = get_conn()

    for year in years:
        df = statcast_batter_exitvelo_barrels(year, minBBE=minBBE)
        df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
        df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
        df['year'] = year
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_batter_exitvelo_barrels_to_rds()

[2025-04-10 12:39:20.577613] ✅ Uploaded 915 rows to batter_exitvelo_barrels
[2025-04-10 12:39:20.877341] ✅ Uploaded 909 rows to batter_exitvelo_barrels
[2025-04-10 12:39:21.183962] ✅ Uploaded 905 rows to batter_exitvelo_barrels
[2025-04-10 12:39:21.486642] ✅ Uploaded 911 rows to batter_exitvelo_barrels
[2025-04-10 12:39:21.748070] ✅ Uploaded 910 rows to batter_exitvelo_barrels
[2025-04-10 12:39:22.080079] ✅ Uploaded 576 rows to batter_exitvelo_barrels
[2025-04-10 12:39:22.328554] ✅ Uploaded 945 rows to batter_exitvelo_barrels
[2025-04-10 12:39:22.543150] ✅ Uploaded 685 rows to batter_exitvelo_barrels
[2025-04-10 12:39:22.777006] ✅ Uploaded 651 rows to batter_exitvelo_barrels
[2025-04-10 12:39:23.130344] ✅ Uploaded 647 rows to batter_exitvelo_barrels


#### Uploading Batter Expected Stats to RDS

In [0]:
def upload_batters_expected_stats_to_rds(years=[x for x in range(2015,2025)], minPA=1):
    table = 'batter_expected_stats'

    conn = get_conn()

    for year in years:
        df = statcast_batter_expected_stats(year, minPA=minPA)
        df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
        df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_batters_expected_stats_to_rds()

[2025-04-10 13:29:31.115146] ✅ Uploaded 964 rows to batter_expected_stats
[2025-04-10 13:29:31.382576] ✅ Uploaded 969 rows to batter_expected_stats
[2025-04-10 13:29:31.589687] ✅ Uploaded 957 rows to batter_expected_stats
[2025-04-10 13:29:31.801515] ✅ Uploaded 990 rows to batter_expected_stats
[2025-04-10 13:29:32.104509] ✅ Uploaded 990 rows to batter_expected_stats
[2025-04-10 13:29:32.294103] ✅ Uploaded 581 rows to batter_expected_stats
[2025-04-10 13:29:32.609064] ✅ Uploaded 1049 rows to batter_expected_stats
[2025-04-10 13:29:32.956839] ✅ Uploaded 693 rows to batter_expected_stats
[2025-04-10 13:29:33.152848] ✅ Uploaded 656 rows to batter_expected_stats
[2025-04-10 13:29:33.325516] ✅ Uploaded 651 rows to batter_expected_stats


#### Uploading Batter Percentile Ranks to RDS

In [0]:
def upload_batter_percentile_ranks_to_rds(years=[x for x in range(2015,2025)]):
    table = 'batter_percentile_ranks'

    conn = get_conn()

    for year in years:
        df = statcast_batter_percentile_ranks(year)
        df[['last_name', 'first_name']] = df['player_name'].str.split(', ', expand=True)
        df.drop(columns = ['player_name', 'player_id'], inplace=True)
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_batter_percentile_ranks_to_rds()

[2025-04-10 13:17:11.068681] ✅ Uploaded 683 rows to batter_percentile_ranks
[2025-04-10 13:17:11.585708] ✅ Uploaded 693 rows to batter_percentile_ranks
[2025-04-10 13:17:11.945764] ✅ Uploaded 678 rows to batter_percentile_ranks
[2025-04-10 13:17:12.434170] ✅ Uploaded 678 rows to batter_percentile_ranks
[2025-04-10 13:17:12.899660] ✅ Uploaded 688 rows to batter_percentile_ranks
[2025-04-10 13:17:13.358540] ✅ Uploaded 507 rows to batter_percentile_ranks
[2025-04-10 13:17:13.737113] ✅ Uploaded 684 rows to batter_percentile_ranks
[2025-04-10 13:17:14.170356] ✅ Uploaded 628 rows to batter_percentile_ranks
[2025-04-10 13:17:14.531413] ✅ Uploaded 614 rows to batter_percentile_ranks
[2025-04-10 13:17:14.911791] ✅ Uploaded 606 rows to batter_percentile_ranks


#### Upload Batter Pitcher Arsenal to RDS

In [0]:
def upload_batter_pitcher_arsenal_to_rds(years=[x for x in range(2015,2025)], minPA=1):
    table = 'batter_pitcher_arsenal'

    conn = get_conn()

    for year in years:
        df = statcast_batter_pitch_arsenal(year, minPA=minPA)
        if len(df) > 0:
            df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
            df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
            fast_copy_to_sql(df, table, conn)

    conn.close()

upload_batter_pitcher_arsenal_to_rds()

[2025-04-10 13:39:38.681926] ✅ Uploaded 6024 rows to batter_pitcher_arsenal
[2025-04-10 13:39:39.911732] ✅ Uploaded 5936 rows to batter_pitcher_arsenal
[2025-04-10 13:39:41.182793] ✅ Uploaded 6059 rows to batter_pitcher_arsenal
[2025-04-10 13:39:42.205851] ✅ Uploaded 4108 rows to batter_pitcher_arsenal
[2025-04-10 13:39:43.942219] ✅ Uploaded 6329 rows to batter_pitcher_arsenal
[2025-04-10 13:39:45.161918] ✅ Uploaded 5362 rows to batter_pitcher_arsenal
[2025-04-10 13:39:46.342667] ✅ Uploaded 5259 rows to batter_pitcher_arsenal
[2025-04-10 13:39:47.705344] ✅ Uploaded 5358 rows to batter_pitcher_arsenal


### Other Statcast Pitcher Data

#### Uploading Pitcher Exit Velocity and Barrels Data to RDS

In [0]:
def upload_pitcher_exitvelo_barrels_to_rds(years=[x for x in range(2015,2025)], minBBE=1):
    table = 'pitcher_exitvelo_barrels'

    conn = get_conn()

    for year in years:
        df = statcast_pitcher_exitvelo_barrels(year, minBBE=minBBE)
        df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
        df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_pitcher_exitvelo_barrels_to_rds()

[2025-04-10 13:53:53.183953] ✅ Uploaded 734 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:53.451672] ✅ Uploaded 742 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:53.653866] ✅ Uploaded 752 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:53.877238] ✅ Uploaded 797 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:54.095470] ✅ Uploaded 827 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:54.302580] ✅ Uploaded 734 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:54.579861] ✅ Uploaded 906 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:54.818405] ✅ Uploaded 871 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:55.045697] ✅ Uploaded 863 rows to pitcher_exitvelo_barrels
[2025-04-10 13:53:55.279390] ✅ Uploaded 854 rows to pitcher_exitvelo_barrels


#### Uploading Pitcher Expected Stats to RDS

In [0]:
def upload_pitcher_expected_stats_to_rds(years=[x for x in range(2015,2025)], minPA=1):
    table = 'pitcher_expected_stats'

    conn = get_conn()

    for year in years:
        df = statcast_pitcher_expected_stats(year, minPA=minPA)
        df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
        df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_pitcher_expected_stats_to_rds()

[2025-04-10 14:03:12.248820] ✅ Uploaded 735 rows to pitcher_expected_stats
[2025-04-10 14:03:12.895673] ✅ Uploaded 742 rows to pitcher_expected_stats
[2025-04-10 14:03:13.390627] ✅ Uploaded 755 rows to pitcher_expected_stats
[2025-04-10 14:03:13.949604] ✅ Uploaded 799 rows to pitcher_expected_stats
[2025-04-10 14:03:14.671685] ✅ Uploaded 831 rows to pitcher_expected_stats
[2025-04-10 14:03:15.200881] ✅ Uploaded 735 rows to pitcher_expected_stats
[2025-04-10 14:03:15.768183] ✅ Uploaded 909 rows to pitcher_expected_stats
[2025-04-10 14:03:16.304792] ✅ Uploaded 871 rows to pitcher_expected_stats
[2025-04-10 14:03:16.852066] ✅ Uploaded 863 rows to pitcher_expected_stats
[2025-04-10 14:03:17.402269] ✅ Uploaded 855 rows to pitcher_expected_stats


#### Upload Pitcher Pitcher Arsenal to RDS

In [0]:
def upload_pitcher_pitch_arsenal_to_rds(years=[x for x in range(2015,2025)], minP=1):
    table = 'pitcher_pitch_arsenal'

    conn = get_conn()

    for year in years:
        df1 = statcast_pitcher_pitch_arsenal(year, minP=1, arsenal_type = "n_")
        df2 = statcast_pitcher_pitch_arsenal(year, minP=1, arsenal_type = "avg_speed")
        df3 = statcast_pitcher_pitch_arsenal(year, minP=1, arsenal_type = "avg_spin")
        df = df1.merge(df2, on=["last_name, first_name", "pitcher"]).merge(df3, on=["last_name, first_name", "pitcher"])
        df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
        df.drop(columns=['last_name, first_name', 'pitcher'], inplace=True)
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_pitcher_pitch_arsenal_to_rds()

[2025-04-10 14:08:17.199630] ✅ Uploaded 735 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:18.940041] ✅ Uploaded 745 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:20.790947] ✅ Uploaded 749 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:22.189427] ✅ Uploaded 786 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:23.499340] ✅ Uploaded 819 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:25.149384] ✅ Uploaded 723 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:26.556765] ✅ Uploaded 858 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:27.786467] ✅ Uploaded 813 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:29.509626] ✅ Uploaded 805 rows to pitcher_pitch_arsenal
[2025-04-10 14:08:30.877278] ✅ Uploaded 806 rows to pitcher_pitch_arsenal


#### Uploading Pitcher Arsenal Stats to RDS

In [0]:
def upload_pitcher_arsenal_stats_to_rds(years=[x for x in range(2015,2025)], minPA=1):
    table = 'pitcher_arsenal_stats'

    conn = get_conn()

    for year in years:
        df = statcast_pitcher_arsenal_stats(year, minPA=minPA)
        if len(df) > 0:
            df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
            df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
            fast_copy_to_sql(df, table, conn)

    conn.close()

upload_pitcher_arsenal_stats_to_rds()

[2025-04-10 14:13:26.356425] ✅ Uploaded 3063 rows to pitcher_arsenal_stats
[2025-04-10 14:13:27.414497] ✅ Uploaded 3078 rows to pitcher_arsenal_stats
[2025-04-10 14:13:28.306892] ✅ Uploaded 3164 rows to pitcher_arsenal_stats
[2025-04-10 14:13:29.120048] ✅ Uploaded 2689 rows to pitcher_arsenal_stats
[2025-04-10 14:13:29.917842] ✅ Uploaded 3381 rows to pitcher_arsenal_stats
[2025-04-10 14:13:30.873992] ✅ Uploaded 3271 rows to pitcher_arsenal_stats
[2025-04-10 14:13:32.862272] ✅ Uploaded 3379 rows to pitcher_arsenal_stats
[2025-04-10 14:13:33.782149] ✅ Uploaded 3469 rows to pitcher_arsenal_stats


#### Upload Pitcher Percentile Ranks to RDS

In [0]:
def upload_pitcher_percentile_ranks_to_rds(years=[x for x in range(2015,2025)]):
    table = 'pitcher_percentile_ranks'

    conn = get_conn()

    for year in years:
        df = statcast_pitcher_percentile_ranks(year)
        df[['last_name', 'first_name']] = df['player_name'].str.split(', ', expand=True)
        df.drop(columns=['player_name', 'player_id'], inplace=True)
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_pitcher_percentile_ranks_to_rds()

[2025-04-10 14:21:34.507703] ✅ Uploaded 649 rows to pitcher_percentile_ranks
[2025-04-10 14:21:34.980567] ✅ Uploaded 657 rows to pitcher_percentile_ranks
[2025-04-10 14:21:35.395632] ✅ Uploaded 676 rows to pitcher_percentile_ranks
[2025-04-10 14:21:35.952878] ✅ Uploaded 695 rows to pitcher_percentile_ranks
[2025-04-10 14:21:36.209304] ✅ Uploaded 705 rows to pitcher_percentile_ranks
[2025-04-10 14:21:36.737732] ✅ Uploaded 585 rows to pitcher_percentile_ranks
[2025-04-10 14:21:37.201315] ✅ Uploaded 735 rows to pitcher_percentile_ranks
[2025-04-10 14:21:37.600583] ✅ Uploaded 704 rows to pitcher_percentile_ranks
[2025-04-10 14:21:38.072052] ✅ Uploaded 707 rows to pitcher_percentile_ranks
[2025-04-10 14:21:38.391704] ✅ Uploaded 696 rows to pitcher_percentile_ranks


#### Uploading Pitch Movement to RDS

In [0]:
def statcast_pitcher_pitch_movement(year: int, minP: Union[int, str] = "q", pitch_type: str = "FF") -> pd.DataFrame:
    pitch_type = norm_pitch_code(pitch_type)
    url = f"https://baseballsavant.mlb.com/leaderboard/pitch-movement?year={year}&team=&min={minP}&pitch_type={pitch_type}&hand=&x=pitcher_break_x_hidden&z=pitcher_break_z_hidden&csv=true"
    res = requests.get(url, timeout=None).content
    data = pd.read_csv(StringIO(res.decode('utf-8')))
    data = sanitize_statcast_columns(data)
    return data

In [0]:
def upload_pitch_movement_to_rds(years=[x for x in range(2015,2025)], pitches=pitch_codes, minP=1):
    table = 'pitch_movement'

    conn = get_conn()

    for year in years:
        for pitch in pitches:
            df = statcast_pitcher_pitch_movement(year, minP=minP, pitch_type=pitch)
            if len(df) > 0:
                df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
                df.drop(columns=['last_name, first_name', 'pitcher_id'], inplace=True)
                fast_copy_to_sql(df, table, conn)

    conn.close()

upload_pitch_movement_to_rds()

[2025-04-10 15:54:31.235721] ✅ Uploaded 680 rows to pitch_movement
[2025-04-10 15:54:31.542141] ✅ Uploaded 499 rows to pitch_movement
[2025-04-10 15:54:31.800498] ✅ Uploaded 616 rows to pitch_movement
[2025-04-10 15:54:31.999624] ✅ Uploaded 208 rows to pitch_movement
[2025-04-10 15:54:32.415186] ✅ Uploaded 1 rows to pitch_movement
[2025-04-10 15:54:32.781210] ✅ Uploaded 2 rows to pitch_movement
[2025-04-10 15:54:33.247737] ✅ Uploaded 1 rows to pitch_movement
[2025-04-10 15:54:33.682605] ✅ Uploaded 573 rows to pitch_movement
[2025-04-10 15:54:34.072523] ✅ Uploaded 542 rows to pitch_movement
[2025-04-10 15:54:34.320516] ✅ Uploaded 51 rows to pitch_movement
[2025-04-10 15:54:35.272594] ✅ Uploaded 7 rows to pitch_movement
[2025-04-10 15:54:35.521038] ✅ Uploaded 3 rows to pitch_movement
[2025-04-10 15:54:36.261148] ✅ Uploaded 679 rows to pitch_movement
[2025-04-10 15:54:36.663356] ✅ Uploaded 494 rows to pitch_movement
[2025-04-10 15:54:37.041265] ✅ Uploaded 623 rows to pitch_movement
[2025-

#### Upload Pitcher Active Spin to RDS

In [0]:
def statcast_pitcher_active_spin(year: int, minP: int = 250, _type: str = 'spin-based') -> pd.DataFrame:
    url = f"https://baseballsavant.mlb.com/leaderboard/active-spin?year={year}_{_type}&min={minP}&hand=&csv=true"
    res = requests.get(url, timeout=None).content
    if res and '<html' in res.decode('utf-8'):
        # This did no go as planned. Statcast redirected us back to HTML :(
        if _type == 'spin-based':
            warnings.warn(f'Could not get active spin results for year {year} that are "spin-based". Trying to get the older "observed" results.')
            return statcast_pitcher_active_spin(year, minP, 'observed')
        
        warnings.warn("Statcast did not return any active spin results for the query provided.")
        return pd.DataFrame()

    data = pd.read_csv(StringIO(res.decode('utf-8')))
    if _type == 'spin-based' and (data is None or data.empty):
        return statcast_pitcher_active_spin(year, minP, 'observed')

    data = sanitize_statcast_columns(data)
    return data

In [0]:
def upload_active_spin_to_rds(years=[x for x in range(2015, 2025)], minP=1):
    table = 'active_spin'

    conn = get_conn()

    for year in years:
        df = statcast_pitcher_active_spin(year, minP=minP)
        df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
        df.drop(columns=['last_name, first_name'], inplace=True)
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_active_spin_to_rds()

[2025-04-10 16:03:44.879374] ✅ Uploaded 632 rows to active_spin
[2025-04-10 16:03:45.475358] ✅ Uploaded 638 rows to active_spin
[2025-04-10 16:03:46.283977] ✅ Uploaded 654 rows to active_spin
[2025-04-10 16:03:46.916684] ✅ Uploaded 685 rows to active_spin
[2025-04-10 16:03:47.630685] ✅ Uploaded 691 rows to active_spin
[2025-04-10 16:03:48.183436] ✅ Uploaded 562 rows to active_spin
[2025-04-10 16:03:48.556514] ✅ Uploaded 733 rows to active_spin
[2025-04-10 16:03:48.898198] ✅ Uploaded 707 rows to active_spin
[2025-04-10 16:03:49.283275] ✅ Uploaded 703 rows to active_spin
[2025-04-10 16:03:49.650873] ✅ Uploaded 708 rows to active_spin


### Statcast Fielding Data

#### Upload OAA to RDS

In [0]:
def upload_oaa_to_rds(years=[x for x in range(2015,2025)], positions=[x for x in range(3, 10)], min_att=1):
    table = 'oaa'

    conn = get_conn()

    for year in years:
        for position in positions:
            df = statcast_outs_above_average(year, position, min_att=min_att)
            if len(df) > 0:
                df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
                df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
                fast_copy_to_sql(df, table, conn)

    conn.close()

upload_oaa_to_rds()

[2025-04-10 16:12:30.065005] ✅ Uploaded 160 rows to oaa
[2025-04-10 16:12:31.036551] ✅ Uploaded 150 rows to oaa
[2025-04-10 16:12:31.943943] ✅ Uploaded 158 rows to oaa
[2025-04-10 16:12:32.950037] ✅ Uploaded 121 rows to oaa
[2025-04-10 16:12:34.704343] ✅ Uploaded 233 rows to oaa
[2025-04-10 16:12:35.639581] ✅ Uploaded 153 rows to oaa
[2025-04-10 16:12:36.905547] ✅ Uploaded 197 rows to oaa
[2025-04-10 16:12:37.803300] ✅ Uploaded 155 rows to oaa
[2025-04-10 16:12:38.640735] ✅ Uploaded 154 rows to oaa
[2025-04-10 16:12:39.536073] ✅ Uploaded 163 rows to oaa
[2025-04-10 16:12:41.596909] ✅ Uploaded 126 rows to oaa
[2025-04-10 16:12:42.583149] ✅ Uploaded 223 rows to oaa
[2025-04-10 16:12:43.634564] ✅ Uploaded 159 rows to oaa
[2025-04-10 16:12:44.505073] ✅ Uploaded 182 rows to oaa
[2025-04-10 16:12:45.495023] ✅ Uploaded 163 rows to oaa
[2025-04-10 16:12:46.411597] ✅ Uploaded 169 rows to oaa
[2025-04-10 16:12:47.295814] ✅ Uploaded 173 rows to oaa
[2025-04-10 16:12:48.328029] ✅ Uploaded 121 rows

#### Upload Directional OAA to RDS

In [0]:
def upload_directional_oaa_to_rds(years=[x for x in range(2015,2025)], min_opp=1):
    table = 'directional_oaa'

    conn = get_conn()

    for year in years:
        df = statcast_outfield_directional_oaa(year, min_opp=min_opp)
        if len(df) > 0:
            df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
            df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
            fast_copy_to_sql(df, table, conn)

    conn.close()

upload_directional_oaa_to_rds()

[2025-04-10 16:21:42.180217] ✅ Uploaded 310 rows to directional_oaa
[2025-04-10 16:21:42.546688] ✅ Uploaded 309 rows to directional_oaa
[2025-04-10 16:21:42.920106] ✅ Uploaded 295 rows to directional_oaa
[2025-04-10 16:21:43.292730] ✅ Uploaded 317 rows to directional_oaa
[2025-04-10 16:21:43.626920] ✅ Uploaded 266 rows to directional_oaa
[2025-04-10 16:21:43.990453] ✅ Uploaded 340 rows to directional_oaa
[2025-04-10 16:21:44.211154] ✅ Uploaded 337 rows to directional_oaa
[2025-04-10 16:21:44.514064] ✅ Uploaded 312 rows to directional_oaa
[2025-04-10 16:21:44.804837] ✅ Uploaded 302 rows to directional_oaa


#### Upload Outfield Catch Probability to RDS

In [0]:
def upload_outfield_catch_prob_to_rds(years=[x for x in range(2015, 2025)], min_opp=1):
    table = 'outfield_catch_prob'

    conn = get_conn()

    for year in years:
        df = statcast_outfield_catch_prob(year, min_opp=min_opp)
        if len(df) > 0:
            df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
            df.drop(columns=['last_name, first_name', 'player_id'], inplace=True)
            fast_copy_to_sql(df, table, conn)

    conn.close()

upload_outfield_catch_prob_to_rds()

[2025-04-10 16:40:50.656170] ✅ Uploaded 310 rows to outfield_catch_prob
[2025-04-10 16:40:51.035359] ✅ Uploaded 309 rows to outfield_catch_prob
[2025-04-10 16:40:51.441438] ✅ Uploaded 295 rows to outfield_catch_prob
[2025-04-10 16:40:51.709812] ✅ Uploaded 317 rows to outfield_catch_prob
[2025-04-10 16:40:52.012840] ✅ Uploaded 266 rows to outfield_catch_prob
[2025-04-10 16:40:52.417101] ✅ Uploaded 340 rows to outfield_catch_prob
[2025-04-10 16:40:52.728134] ✅ Uploaded 337 rows to outfield_catch_prob
[2025-04-10 16:40:53.397593] ✅ Uploaded 312 rows to outfield_catch_prob
[2025-04-10 16:40:53.669207] ✅ Uploaded 302 rows to outfield_catch_prob


#### Upload Outfielder Jump to RDS

In [0]:
def upload_of_jump_to_rds(years=[x for x in range(2015, 2025)], min_att=1):
    table = 'outfield_jump'

    conn = get_conn()

    for year in years:
        df = statcast_outfielder_jump(year, min_att=min_att)

        if len(df) > 0:
            df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
            df.drop(columns=['last_name, first_name'], inplace=True)
            fast_copy_to_sql(df, table, conn)

    conn.close()

upload_of_jump_to_rds()

[2025-04-10 16:48:37.309550] ✅ Uploaded 211 rows to outfield_jump
[2025-04-10 16:48:37.644095] ✅ Uploaded 218 rows to outfield_jump
[2025-04-10 16:48:37.942750] ✅ Uploaded 220 rows to outfield_jump
[2025-04-10 16:48:38.243191] ✅ Uploaded 235 rows to outfield_jump
[2025-04-10 16:48:38.466473] ✅ Uploaded 161 rows to outfield_jump
[2025-04-10 16:48:38.801701] ✅ Uploaded 235 rows to outfield_jump
[2025-04-10 16:48:39.105223] ✅ Uploaded 239 rows to outfield_jump
[2025-04-10 16:48:39.328625] ✅ Uploaded 233 rows to outfield_jump
[2025-04-10 16:48:39.563821] ✅ Uploaded 219 rows to outfield_jump


#### Upload Catcher Pop Time to RDS

In [0]:
def upload_catcher_poptime_to_rds(years=[x for x in range(2015,2025)], min_2b_att=0, min_3b_att=0):
    table = 'catcher_poptime'

    conn = get_conn()

    for year in years:
        df = statcast_catcher_poptime(year, min_2b_att=min_2b_att, min_3b_att=min_3b_att)
        if len(df) > 0:
            df[['last_name', 'first_name']] = df['catcher'].str.split(', ', expand=True)
            df.drop(columns=['catcher', 'player_id', 'team_id'], inplace=True)
            fast_copy_to_sql(df, table, conn)

    conn.close()

upload_catcher_poptime_to_rds()

[2025-04-10 17:46:24.411548] ✅ Uploaded 108 rows to catcher_poptime
[2025-04-10 17:46:24.607873] ✅ Uploaded 101 rows to catcher_poptime
[2025-04-10 17:46:24.769816] ✅ Uploaded 109 rows to catcher_poptime
[2025-04-10 17:46:24.961863] ✅ Uploaded 114 rows to catcher_poptime
[2025-04-10 17:46:25.090406] ✅ Uploaded 111 rows to catcher_poptime
[2025-04-10 17:46:25.245600] ✅ Uploaded 99 rows to catcher_poptime
[2025-04-10 17:46:25.395259] ✅ Uploaded 116 rows to catcher_poptime
[2025-04-10 17:46:25.573322] ✅ Uploaded 120 rows to catcher_poptime
[2025-04-10 17:46:25.777520] ✅ Uploaded 102 rows to catcher_poptime
[2025-04-10 17:46:25.942648] ✅ Uploaded 100 rows to catcher_poptime


#### Upload Catcher Framing to RDS

In [0]:
def upload_catcher_framing_to_rds(years=[x for x in range(2015,2025)], min_called_p=1):
    table = 'catcher_framing'

    conn = get_conn()

    for year in years:
        df = statcast_catcher_framing(year, min_called_p=min_called_p)
        df.drop(columns=['player_id'], inplace=True)
        fast_copy_to_sql(df, table, conn)

    conn.close()

upload_catcher_framing_to_rds()

[2025-04-10 17:04:10.105949] ✅ Uploaded 110 rows to catcher_framing
[2025-04-10 17:04:10.547695] ✅ Uploaded 105 rows to catcher_framing
[2025-04-10 17:04:10.886521] ✅ Uploaded 114 rows to catcher_framing
[2025-04-10 17:04:11.139122] ✅ Uploaded 117 rows to catcher_framing
[2025-04-10 17:04:11.487263] ✅ Uploaded 113 rows to catcher_framing
[2025-04-10 17:04:11.786830] ✅ Uploaded 103 rows to catcher_framing
[2025-04-10 17:04:12.038912] ✅ Uploaded 117 rows to catcher_framing
[2025-04-10 17:04:13.232993] ✅ Uploaded 121 rows to catcher_framing
[2025-04-10 17:04:13.501722] ✅ Uploaded 103 rows to catcher_framing
[2025-04-10 17:04:13.792847] ✅ Uploaded 101 rows to catcher_framing


#### Upload Fielding Run Value to RDS

In [0]:
def statcast_fielding_run_value(year: int, pos: Union[int, str], min_inn: int = 100) -> pd.DataFrame:
	pos = norm_positions(pos)
	url = f"https://baseballsavant.mlb.com/leaderboard/fielding-run-value?year={year}&min={min_inn}&pos={pos}&roles=&viz=show&csv=true"
	res = requests.get(url, timeout=None).content
	data = pd.read_csv(StringIO(res.decode('utf-8')))
	data = sanitize_statcast_columns(data)
	return data

In [0]:
def upload_fielding_run_value_to_rds(years=[x for x in range(2016, 2025)], positions=[x for x in range(2, 10)], min_inn=1):
    table = 'fielding_run_value'

    conn = get_conn()

    for year in years:
        for position in positions:
            df = statcast_fielding_run_value(year, pos=position, min_inn=min_inn)
            if len(df) > 0:
                df[['last_name', 'first_name']] = df['player_name'].str.split(', ', expand=True)
                df['position'] = position
                df.drop(columns=['player_name', 'player_id', 'team_id'], inplace=True)
                fast_copy_to_sql(df, table, conn)

    conn.close()

upload_fielding_run_value_to_rds()

[2025-04-10 17:08:15.661158] ✅ Uploaded 104 rows to fielding_run_value
[2025-04-10 17:08:16.833410] ✅ Uploaded 178 rows to fielding_run_value
[2025-04-10 17:08:18.530070] ✅ Uploaded 157 rows to fielding_run_value
[2025-04-10 17:08:19.514760] ✅ Uploaded 170 rows to fielding_run_value
[2025-04-10 17:08:20.822690] ✅ Uploaded 125 rows to fielding_run_value
[2025-04-10 17:08:22.176580] ✅ Uploaded 251 rows to fielding_run_value
[2025-04-10 17:08:23.091147] ✅ Uploaded 160 rows to fielding_run_value
[2025-04-10 17:08:24.104730] ✅ Uploaded 214 rows to fielding_run_value
[2025-04-10 17:08:25.194636] ✅ Uploaded 112 rows to fielding_run_value
[2025-04-10 17:08:26.301414] ✅ Uploaded 183 rows to fielding_run_value
[2025-04-10 17:08:27.682666] ✅ Uploaded 158 rows to fielding_run_value
[2025-04-10 17:08:28.640388] ✅ Uploaded 174 rows to fielding_run_value
[2025-04-10 17:08:30.111688] ✅ Uploaded 128 rows to fielding_run_value
[2025-04-10 17:08:31.350593] ✅ Uploaded 241 rows to fielding_run_value
[2025-

### Statcast Running Data

#### Upload Sprint Speed Data to RDS

In [0]:
def upload_sprint_speed_to_rds(years=[x for x in range(2015, 2025)], min_opp=1):
    table = 'sprint_speed'

    conn = get_conn()

    for year in years:
        df = statcast_sprint_speed(year, min_opp=min_opp)
        if len(df) > 0:
            df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
            df.drop(columns=['last_name, first_name', 'player_id', 'team_id', 'team'], inplace=True)
            fast_copy_to_sql(df, table, conn)

    conn.close()

upload_sprint_speed_to_rds()

[2025-04-10 17:15:51.188445] ✅ Uploaded 586 rows to sprint_speed
[2025-04-10 17:15:51.613138] ✅ Uploaded 590 rows to sprint_speed
[2025-04-10 17:15:51.878026] ✅ Uploaded 584 rows to sprint_speed
[2025-04-10 17:15:52.238887] ✅ Uploaded 587 rows to sprint_speed
[2025-04-10 17:15:52.570561] ✅ Uploaded 602 rows to sprint_speed
[2025-04-10 17:15:52.892444] ✅ Uploaded 507 rows to sprint_speed
[2025-04-10 17:15:53.232958] ✅ Uploaded 602 rows to sprint_speed
[2025-04-10 17:15:53.548829] ✅ Uploaded 628 rows to sprint_speed
[2025-04-10 17:15:53.928221] ✅ Uploaded 614 rows to sprint_speed
[2025-04-10 17:15:54.188804] ✅ Uploaded 606 rows to sprint_speed


#### Upload Running Splits to RDS

In [0]:
def upload_running_splits_to_rds(years=[x for x in range(2015, 2025)], min_opp=1):
    table = 'running_splits'

    conn = get_conn()

    for year in years:
        df = statcast_running_splits(year, min_opp=min_opp)
        if len(df) > 0:
            df[['last_name', 'first_name']] = df['last_name, first_name'].str.split(', ', expand=True)
            df.drop(columns=['last_name, first_name', 'name_abbrev', 'player_id', 'team_id'], inplace=True)
            fast_copy_to_sql(df, table, conn)

    conn.close()

upload_running_splits_to_rds()

[2025-04-10 17:21:39.733696] ✅ Uploaded 504 rows to running_splits
[2025-04-10 17:21:40.138359] ✅ Uploaded 536 rows to running_splits
[2025-04-10 17:21:40.581437] ✅ Uploaded 537 rows to running_splits
[2025-04-10 17:21:41.055685] ✅ Uploaded 533 rows to running_splits
[2025-04-10 17:21:41.544470] ✅ Uploaded 548 rows to running_splits
[2025-04-10 17:21:41.871670] ✅ Uploaded 359 rows to running_splits
[2025-04-10 17:21:42.179222] ✅ Uploaded 522 rows to running_splits
[2025-04-10 17:21:42.631997] ✅ Uploaded 574 rows to running_splits
[2025-04-10 17:21:43.114100] ✅ Uploaded 555 rows to running_splits
[2025-04-10 17:21:43.541394] ✅ Uploaded 533 rows to running_splits


### Saving Training Data

In [0]:
training_data = pd.read_csv('training_examples.csv')

df = spark.createDataFrame(training_data)
df.write.mode("overwrite").parquet("***")

training_data.to_parquet(train_data_directory, index=False)